# Agência Nacional de Energia Elétrica (ANEEL)

O site da ANEEL apresenta uma sessão de [**Informações Geográficas**](https://www.aneel.gov.br/informacoes-geograficas). Analisando o material, costata-se que eles utilizam a estrutura do _ArcGIS Server_, em um portal denominado _Sistema de Informações Georreferenciadas do Setor Elétrico (SIGEL)_.

- https://sigel.aneel.gov.br/arcgis/rest/

<br>

---

## Siglas

<br>

A instituição tem várias siglas que estão apresentadas nos dados do ArcGIS.

**Superintendências**

- SRM: Superintendência de Regulação Econômica e Estudos de Mercado
- SMA: Superintendência de Mediação Administrativa, Ouvidoria Setorial e Participação Pública
- SFE: Superintendência de Fiscalização dos Serviços de Eletricidade
- SFG: Superintendência de Fiscalização dos Serviços de Geração

<br>

**Outras**

- SIPH: Sistema de Informações do Potencial Hidroelétrico
- GGT: Gestão Geoespacializada da Transmissão
- SGO: Sistema de Gestão de Ouvidoria
- IASC: Índice ANEEL de Satisfação do Consumidor
- UFV: Centrais Geradoras Fotovoltaicas
- EOL: Usinas Eólicas
- UTN: Usina Eletronuclear
- UTE: Usinas Termelétricas
- PCH: Pequenas Centrais Hidrelétricas
- AHE: Aproveitamentos Hidrelétricos
- UHE: Usinas Hidrelétricas
- SKATE

<br>

---

## _Download_ de Dados

Por meio do acesso ao site de _download_ dos dados, é possível observar a interface e _layers_ disponíveis. Observou-se que são os mesmos _layers_ disponíveis na pasta "Portal" do _webservice_.

![ANEEL](https://i.imgur.com/JoQs2ZT.png)


Para os pacotes que usam python, é necessário

In [1]:
#!pip3 install arcgis

Definir a variável de ambiente `RESTAPI_USE_ARCPY` como `FALSE` é necessário para evitar que a biblioteca `restapi` mande mensagens de erro ou tente usar o `ArcPy`, que só está disponível para quem tem licença da ESRI.


In [2]:
import os

os.environ['RESTAPI_USE_ARCPY'] = 'FALSE'

In [3]:
import warnings

import requests
import restapi
from restapi import NAME, SERVICES, TYPE, ArcServer

import open_geodata as geo

In [4]:
import json
import pprint
import tempfile
from pathlib import Path
# from arcgis.raster.functions import *
import geopandas as gpd

In [5]:
warnings.filterwarnings("ignore")

In [6]:
session = requests.Session()
client = restapi.RequestClient(session)
restapi.set_request_client(client)

In [7]:
# connect to esri's sample server 6
url = 'https://sigel.aneel.gov.br/arcgis/rest/services'
url

'https://sigel.aneel.gov.br/arcgis/rest/services'

In [8]:
# Connect to restapi.ArcServer instance
ags = restapi.ArcServer(url)
ags

<ArcServer: "sigel.aneel.gov.br" ("arcgis")>

<br>

Com o uso do rest

In [9]:
for x in ags.list_services():
    print(x, type(x))

https://sigel.aneel.gov.br/arcgis/rest/services/CartasTopograficas/MapServer <class 'str'>
https://sigel.aneel.gov.br/arcgis/rest/services/check_data/MapServer <class 'str'>
https://sigel.aneel.gov.br/arcgis/rest/services/Features_DEEP_UFV2/FeatureServer <class 'str'>
https://sigel.aneel.gov.br/arcgis/rest/services/Features_DEEP_UFV2/MapServer <class 'str'>
https://sigel.aneel.gov.br/arcgis/rest/services/GEO_SMA_SDE_AREA_ATUACAO_DIST_LIM_MUNICIPAL_2018/MapServer <class 'str'>
https://sigel.aneel.gov.br/arcgis/rest/services/Geracao_Distribuida_SIGEL/FeatureServer <class 'str'>
https://sigel.aneel.gov.br/arcgis/rest/services/Geracao_Distribuida_SIGEL/MapServer <class 'str'>
https://sigel.aneel.gov.br/arcgis/rest/services/Geração_Distribuída_MIL1/MapServer <class 'str'>
https://sigel.aneel.gov.br/arcgis/rest/services/Leilões_ANEEL_MIL1/MapServer <class 'str'>
https://sigel.aneel.gov.br/arcgis/rest/services/Mapa_base_validador_UFV_MIL1/MapServer <class 'str'>
https://sigel.aneel.gov.br/arc

In [10]:
for root, services in ags.walk(ignore_folder_auth=True):
    print(f'Pasta: {root}')
    # print('\n'.join(f'- {item}' for item in services))
    for service in services:
        print(f'- {service}')

    print(f'-' * 60)

Pasta: None
- CartasTopograficas/MapServer
- check_data/MapServer
- Features_DEEP_UFV2/FeatureServer
- Features_DEEP_UFV2/MapServer
- GEO_SMA_SDE_AREA_ATUACAO_DIST_LIM_MUNICIPAL_2018/MapServer
- Geracao_Distribuida_SIGEL/FeatureServer
- Geracao_Distribuida_SIGEL/MapServer
- Geração_Distribuída_MIL1/MapServer
- Leilões_ANEEL_MIL1/MapServer
- Mapa_base_validador_UFV_MIL1/MapServer
- Mapa_base_validador_UFV_MIL2/MapServer
- SampleWorldCities/MapServer
- Script_Export_CSV_Imagem/GPServer
- UFV_time/FeatureServer
- UFV_time/MapServer
- Validador_UFV/FeatureServer
- Validador_UFV/MapServer
- Web_Layer_Dash_UFV/FeatureServer
- Web_Layer_Dash_UFV/MapServer
- Web_Layer_UFV_Areas_Especiais/FeatureServer
- Web_Layer_UFV_Areas_Especiais/MapServer
- Web_Layer_UFV_Bases_Validador_UFV/FeatureServer
- Web_Layer_UFV_Bases_Validador_UFV/MapServer
- Web_Layer_UFV_Outras_Bases/FeatureServer
- Web_Layer_UFV_Outras_Bases/MapServer
- WebMap_TimeSlider_SIPH/FeatureServer
- WebMap_TimeSlider_SIPH/MapServer
---

<br>

Obtem detalhes do serviço

In [11]:
# Listas de Tipos de Serviço
ags.featureServices

['https://sigel.aneel.gov.br/arcgis/rest/services/Features_DEEP_UFV2/FeatureServer',
 'https://sigel.aneel.gov.br/arcgis/rest/services/Geracao_Distribuida_SIGEL/FeatureServer',
 'https://sigel.aneel.gov.br/arcgis/rest/services/UFV_time/FeatureServer',
 'https://sigel.aneel.gov.br/arcgis/rest/services/Validador_UFV/FeatureServer',
 'https://sigel.aneel.gov.br/arcgis/rest/services/Web_Layer_Dash_UFV/FeatureServer',
 'https://sigel.aneel.gov.br/arcgis/rest/services/Web_Layer_UFV_Areas_Especiais/FeatureServer',
 'https://sigel.aneel.gov.br/arcgis/rest/services/Web_Layer_UFV_Bases_Validador_UFV/FeatureServer',
 'https://sigel.aneel.gov.br/arcgis/rest/services/Web_Layer_UFV_Outras_Bases/FeatureServer',
 'https://sigel.aneel.gov.br/arcgis/rest/services/WebMap_TimeSlider_SIPH/FeatureServer',
 'https://sigel.aneel.gov.br/arcgis/rest/services/BDIT/Feature_ADS_Area_Desenvolvimento_Subestacao/FeatureServer',
 'https://sigel.aneel.gov.br/arcgis/rest/services/DadosAbertos/AME_2023/FeatureServer',
 '

In [12]:
service = ags.getService(name_or_wildcard='SP')
service.name

'Web_Layer_UFV_Areas_Especiais'

In [13]:
# Shapefile
print(service.name)
print(service.description)

# URL
print(service.url)

# Representação
print(repr(service))

# Formatos
print(service.supportedQueryFormats)

# Path
print(service.servicePath)

# documentInfo
print(service.documentInfo)

# Descrição
print(service.description)

# Informações do datum
print(service.initialExtent)
print(service.spatialReference)

Web_Layer_UFV_Areas_Especiais

https://sigel.aneel.gov.br/arcgis/rest/services/Web_Layer_UFV_Areas_Especiais/FeatureServer
<FeatureService: Web_Layer_UFV_Areas_Especiais/FeatureServer>
JSON
Web_Layer_UFV_Areas_Especiais/FeatureServer
{
  "Author": "",
  "Category": "",
  "Comments": "",
  "Keywords": "UFV",
  "Subject": "",
  "Title": "Map"
}

{
  "spatialReference": {
    "falseM": -100000,
    "falseX": -400,
    "falseY": -400,
    "falseZ": -100000,
    "latestWkid": 4674,
    "mTolerance": 0.001,
    "mUnits": 10000,
    "wkid": 4674,
    "xyTolerance": 8.983152841195212e-09,
    "xyUnits": 999999999.9999999,
    "zTolerance": 0.001,
    "zUnits": 10000
  },
  "xmax": -41.21874898620835,
  "xmin": -61.61934089679155,
  "ymax": 2.446831521132779,
  "ymin": -30.92616843813262
}
4674


In [14]:
service.get_layer_url(name='ZEE')

'https://sigel.aneel.gov.br/arcgis/rest/services/Web_Layer_UFV_Areas_Especiais/FeatureServer/19'

In [15]:
# Seleciona Layer no Serviço
lyr = service.layer(name_or_id=19)

In [16]:
lyr_query = lyr.query(
    where='1=1',
    # Se exceed_limit=True, retorna todos os registros
    # Se exceed_limit=False, retorna apenas os primeiros 1000 registros
    exceed_limit=True,
    # ------------------------------------
    # Número de registros a serem retornados
    # Se records=None, retorna todos os registros
    # records=10,
    # ------------------------------------
    # Option to return a generator with a FeatureSet in chunks of each query group.
    # Use this to avoid memory errors when fetching many features. Defaults to False
    fetch_in_chunks=True,
)

lyr_query.geometryType

'esriGeometryPolygon'

In [17]:
# shp_file
temp_dir = tempfile.TemporaryDirectory()

In [18]:
# Cria o caminho temporário em formato Path
temp_path = Path(temp_dir.name)
temp_path

WindowsPath('C:/Users/michel/AppData/Local/Temp/tmp7_m9qemd')

In [19]:
# ddd
restapi.exportFeatureSet(
    feature_set=lyr_query,
    #
    out_fc=str(temp_path / 'temp.shp'),
)

IndexError: list index out of range

In [ ]:
gdf = gpd.read_file(filename=temp_path / 'temp.shp')

#
gdf.head()

<br>

-----

## ArcGIS


In [ ]:
from arcgis import geometry
from arcgis.geocoding import geocode
from arcgis.gis import GIS

In [ ]:
#url = "https://mapas.agenciapcj.org.br/arcgis/rest/services"

url = service.url
# Connect to the portal
gis = GIS(url)

In [ ]:
# Search for all feature services and feature collections in the portal
items = gis.content.search(query='type: "Feature Service" OR type: "Feature Collection"', max_items=5000)
items